In [60]:
import os
import sys

if "google.colab" in sys.modules:
    # Running in Colab

    !git clone https://github.com/pthengtr/kcw-analytics.git
    !cd /content/kcw-analytics && git pull origin main

    from google.colab import drive
    drive.mount("/content/drive")

    BASE_FOLDER = "/content/drive/Shareddrives"
    BASE_FOLDER_GIT = "/content"
else:
    # Running in local Jupyter
    BASE_FOLDER = r"G:\Shared drives"
    BASE_FOLDER_GIT = r"C:\Users\Windows 11\Notebook"

print("Using folder:", BASE_FOLDER)

fatal: destination path 'kcw-analytics' already exists and is not an empty directory.
From https://github.com/pthengtr/kcw-analytics
 * branch            main       -> FETCH_HEAD
Already up to date.
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Using folder: /content/drive/Shareddrives


In [61]:

folder = f"{BASE_FOLDER}/KCW-Data/kcw_analytics/01_raw"

In [62]:
import os
import pandas as pd

data = {}

for file in os.listdir(folder):
    if file.endswith(".csv"):
        path = os.path.join(folder, file)
        data[file] = pd.read_csv(
            path,
            dtype={
              "BCODE": "string",
              "ITEMNO": "string",
              "BILLNO": "string",
            },
            encoding="utf-8-sig",
            low_memory=False   # stops chunk guessing
        )
        print(f"Loaded: {file} -> {data[file].shape}")



Loaded: raw_inventory_hq_2024.csv -> (4983, 8)
Loaded: raw_syp_pimas_purchase_bills.csv -> (2970, 49)
Loaded: raw_syp_pidet_purchase_lines.csv -> (27770, 41)
Loaded: raw_syp_simas_sales_bills.csv -> (12876, 49)
Loaded: raw_syp_sidet_sales_lines.csv -> (37824, 38)
Loaded: raw_hq_pimas_purchase_bills.csv -> (50274, 49)
Loaded: raw_hq_pidet_purchase_lines.csv -> (154052, 41)
Loaded: raw_hq_sidet_sales_lines.csv -> (733417, 38)
Loaded: raw_hq_icmas_products.csv -> (114950, 94)
Loaded: raw_hq_armas_receivable.csv -> (2227, 20)
Loaded: raw_hq_apmas_payable.csv -> (978, 20)
Loaded: raw_hq_simas_sales_bills.csv -> (275967, 49)
Loaded: raw_hq_pvmas_notes_vouchers.csv -> (13875, 32)


In [63]:

import sys
import importlib

# ensure repo is on path
repo_path = f"{BASE_FOLDER_GIT}/kcw-analytics"
if repo_path not in sys.path:
    sys.path.append(repo_path)

# import the module (NOT individual functions)
import src.kcw.utils as utils

# reload to pick up latest .py changes
importlib.reload(utils)



<module 'src.kcw.utils' from '/content/kcw-analytics/src/kcw/utils.py'>

In [64]:
df_simas = data["raw_hq_simas_sales_bills.csv"].copy()
df_pimas = data["raw_hq_pimas_purchase_bills.csv"].copy()

In [65]:
df_simas.columns

Index(['ID', 'JOURMODE', 'JOURTYPE', 'JOURDATE', 'JOURNO', 'JOURTIME',
       'DEPTNO', 'BOOKNO', 'BILLTYPE', 'BILLDATE', 'BILLTIME', 'BILLNO',
       'LINES', 'TAXIC', 'DISCOUNT', 'DEDUCT', 'BEFORETAX', 'VAT', 'TAX',
       'AFTERTAX', 'EXEMPT', 'SVCCHG', 'WITHHOLD', 'PAID', 'CASHED', 'CASHAMT',
       'CHKAMT', 'DUEAMT', 'PAYSTAT', 'ACCTNO', 'ACCTNAME', 'ADDR1', 'ADDR2',
       'PO', 'SALE', 'RE', 'TERM', 'DUEDATE', 'NOTEDATE', 'NOTENO',
       'VOUCDATE1', 'VOUCNO1', 'VOUCDATE2', 'VOUCNO2', 'POSTED1', 'POSTED2',
       'REMARKS', 'CANCELED', 'DONE'],
      dtype='object')

In [66]:
cutoff = pd.Timestamp.today().replace(day=1)

#cutoff = pd.Timestamp(2026, 2, 1)

YEAR = cutoff.year
MONTH = cutoff.month

cutoff

Timestamp('2026-02-01 00:00:00')

In [67]:
def normalize_acctname(name):
    if pd.isna(name):
        return name

    n = str(name).lower()

    for k in ["lazada", "shopee", "tiktok"]:
        if k in n:
            return f"คุณลูกค้าทั่วไป {k}"

    return name


In [68]:
df = df_simas

df["BILLDATE"] = pd.to_datetime(df["BILLDATE"].astype(str).str.strip(), errors="coerce")
df["VOUCDATE2"] = pd.to_datetime(df["VOUCDATE2"], errors="coerce")

mask_billno = df["BILLNO"].str.contains(r"TD|TR|TAD|CN", na=False)
mask_due_pos = df["DUEAMT"] != 0
mask_billbefore = df["BILLDATE"] < cutoff
mask_paidafter = df["VOUCDATE2"] >= cutoff
mask_null = df["VOUCDATE2"].isna()
mask_cancel = df["CANCELED"].astype(str).str.strip().str.upper() == "N"
mask_vat = pd.to_numeric(df["VAT"], errors="coerce").fillna(0) > 0

df["ACCTNAME"] = df["ACCTNAME"].apply(normalize_acctname)

df_ar_sale = df[mask_billno & mask_due_pos & mask_billbefore & (mask_null | mask_paidafter) & mask_cancel & mask_vat]

df_ar_sale = df_ar_sale.sort_values(by=["ACCTNAME", "BILLDATE"])

In [69]:
df_ar_sale[["BILLNO", "BILLDATE","ACCTNO", "ACCTNAME", "VOUCDATE1", "VOUCDATE2", "PAID","DUEAMT", "VAT"]]

,BILLNO,BILLDATE,ACCTNO,ACCTNAME,VOUCDATE1,VOUCDATE2,PAID,DUEAMT,VAT
270884,TD6901-132,2026-01-31,7ธรพ,คุณ ธีรพงศ์ ฤทธิไกร,NaN,NaT,N,950.0,7.0
270882,TD6901-131,2026-01-31,7บธ,คุณ บุญธรรม สามารถ,NaN,NaT,N,3500.0,7.0
271002,TD6901-135,2026-01-31,7วงศ,คุณ วงษ์สวัสดิ์ จำปาทอง,NaN,NaT,N,1715.0,7.0
271814,TD6901-136,2026-01-31,7รวฒ,คุณ ศิริวุฒิ โสตถิรัตนพันธ์,NaN,NaT,N,6580.0,7.0
270888,TD6901-134,2026-01-31,7สทน,คุณ สุนทร อามาต,NaN,NaT,N,4450.0,7.0
...,...,...,...,...,...,...,...,...,...
270270,TD6901-129,2026-01-31,7ทมซ,บริษัท ไทยไม้ซุง จำกัด (สำนักงานใหญ่),NaN,NaT,N,4520.0,7.0
266426,TD6901-023,2026-01-09,7กช,หจก. ส.กิจชัยคอนกรีต (สาขาที่ 00002),NaN,2026-02-13,Y,3720.0,7.0
249175,TD6810-003,2025-10-01,7จยน,ห้างหุ้นส่วนจำกัด จี.ยูเนี่ยน (สำนักงานใหญ่),NaN,2026-03-04,Y,18860.0,7.0
261324,TD6812-052,2025-12-10,7จยน,ห้างหุ้นส่วนจำกัด จี.ยูเนี่ยน (สำนักงานใหญ่),NaN,2026-03-04,Y,25720.0,7.0


In [70]:
df = df_ar_sale

df_ar_summary = (
    df
    .groupby("ACCTNAME")
    .agg(
        bills=("BILLNO", "nunique"),
        total_amount=("DUEAMT", "sum")
    )
    .reset_index()
)

In [71]:
df_ar_summary

,ACCTNAME,bills,total_amount
0,คุณ ธีรพงศ์ ฤทธิไกร,1,950.00
1,คุณ บุญธรรม สามารถ,1,3500.00
2,คุณ วงษ์สวัสดิ์ จำปาทอง,1,1715.00
3,คุณ ศิริวุฒิ โสตถิรัตนพันธ์,1,6580.00
4,คุณ สุนทร อามาต,1,4450.00
5,คุณ อาบีดีน โด่หลี,1,1940.00
6,คุณ เดชา เกรียงไกร,1,125.00
7,คุณลูกค้าทั่วไป lazada,120,199158.25
8,คุณลูกค้าทั่วไป shopee,43,78900.00
9,คุณลูกค้าทั่วไป tiktok,32,23522.00


In [72]:
df = df_pimas

df["BILLDATE"] = pd.to_datetime(df["BILLDATE"].astype(str).str.strip(), errors="coerce")
df["VOUCDATE2"] = pd.to_datetime(df["VOUCDATE2"], errors="coerce")

mask_due_pos = df["DUEAMT"] != 0
mask_billbefore = df["BILLDATE"] < cutoff
mask_paidafter = df["VOUCDATE2"] >= cutoff
mask_null = df["VOUCDATE2"].isna()
mask_cancel = df["CANCELED"].astype(str).str.strip().str.upper() == "N"
mask_vat = pd.to_numeric(df["VAT"], errors="coerce").fillna(0) > 0

df["ACCTNAME"] = df["ACCTNAME"].apply(normalize_acctname)

df_ap_purchase = df[mask_due_pos & mask_billbefore & (mask_null | mask_paidafter) & mask_cancel & mask_vat]

df_ap_purchase = df_ap_purchase.sort_values(by=["ACCTNAME", "BILLDATE"])

In [73]:
df_ap_purchase[["BILLNO", "BILLDATE","ACCTNO", "ACCTNAME", "VOUCDATE1", "VOUCDATE2", "PAID","DUEAMT", "VAT"]]

,BILLNO,BILLDATE,ACCTNO,ACCTNAME,VOUCDATE1,VOUCDATE2,PAID,DUEAMT,VAT
47886,IV68121304,2025-12-13,7PRW,บจก. ประวิทย์ ออโตพาร์ท อิมปอร์ต-เอ็กซปอร์ต (ส...,NaN,2026-02-24,Y,1897.75,7.0
49197,IV69012611,2026-01-26,7PRW,บจก. ประวิทย์ ออโตพาร์ท อิมปอร์ต-เอ็กซปอร์ต (ส...,NaN,NaT,N,4010.36,7.0
47915,BV6812000012,2025-12-13,7MSC,บจก. มิตซู สแปร์พาร์ทส เอ็กซ์พอร์ทติ้ง (2012) ...,NaN,2026-02-24,Y,4276.04,7.0
48011,BV6812000014,2025-12-16,7MSC,บจก. มิตซู สแปร์พาร์ทส เอ็กซ์พอร์ทติ้ง (2012) ...,NaN,2026-02-24,Y,3950.60,7.0
48466,BV6901000002,2026-01-06,7MSC,บจก. มิตซู สแปร์พาร์ทส เอ็กซ์พอร์ทติ้ง (2012) ...,NaN,NaT,N,8988.75,7.0
...,...,...,...,...,...,...,...,...,...
48902,IV26010234,2026-01-17,7LK,ห้างหุ้นส่วนจำกัด โลหะกิจกลการ อิมปอร์ต (สำนัก...,NaN,NaT,N,4824.51,7.0
48985,IV26010292,2026-01-20,7LK,ห้างหุ้นส่วนจำกัด โลหะกิจกลการ อิมปอร์ต (สำนัก...,NaN,NaT,N,7457.90,7.0
49000,IV26010301,2026-01-21,7LK,ห้างหุ้นส่วนจำกัด โลหะกิจกลการ อิมปอร์ต (สำนัก...,NaN,NaT,N,3356.39,7.0
49002,IV26010319,2026-01-21,7LK,ห้างหุ้นส่วนจำกัด โลหะกิจกลการ อิมปอร์ต (สำนัก...,NaN,NaT,N,2085.38,7.0


In [74]:
df = df_ap_purchase

df_ap_summary = (
    df
    .groupby("ACCTNAME")
    .agg(
        bills=("BILLNO", "nunique"),
        total_amount=("DUEAMT", "sum")
    )
    .reset_index()
)

In [75]:
df_ap_summary

,ACCTNAME,bills,total_amount
0,บจก. ประวิทย์ ออโตพาร์ท อิมปอร์ต-เอ็กซปอร์ต (ส...,2,5908.11
1,บจก. มิตซู สแปร์พาร์ทส เอ็กซ์พอร์ทติ้ง (2012) ...,8,35168.17
2,บจก. เจเจวาย ออโต้ เวิร์คส์ (สำนักงานใหญ่),2,3124.40
3,บจก. เอส ที เค กรุ๊ป 2008 (สำนักงานใหญ่),19,68066.67
4,บจก. เอส.พี.อาร์.วาย ออโต้พาร์ท (สำนักงานใหญ่),27,27429.45
...,...,...,...
88,ห้างหุ้นส่วนจำกัด พี.อี.ออโต้เทรด (สำนักงานใหญ่),5,14990.01
89,ห้างหุ้นส่วนจำกัด รุ่งเรืองทรัพย์ทวี กลการ (สนญ.),1,25252.00
90,ห้างหุ้นส่วนจำกัด ฮ.อาหลั่ยยนต์ (สำนักงานใหญ่),5,23358.10
91,ห้างหุ้นส่วนจำกัด เอส.เค.ที. แมชชีนทูลส์ (สนญ.),38,76230.36


In [76]:
def add_total_row(df, label_col="ACCTNAME", amount_col="total_amount"):
    total_row = pd.DataFrame([{
        label_col: "TOTAL",
        "bills": df["bills"].sum(),
        amount_col: df[amount_col].sum()
    }])

    return pd.concat([df, total_row], ignore_index=True)

In [77]:
def rename_and_keep(df, rename_map):
    cols = [c for c in rename_map.keys() if c in df.columns]
    return df[cols].rename(columns=rename_map)

In [78]:
df_ap_summary = add_total_row(df_ap_summary)
df_ar_summary = add_total_row(df_ar_summary)

In [79]:
rename_map = {
    "ACCTNAME": "ชื่อลูกหนี้",
    "BILLNO": "เลขที่บิล",
    "BILLDATE": "วันที่",
    "DUEAMT": "จำนวน",
    "bills": "จำนวนบิล",
    "total_amount": "ยอดรวม",
}

df_ar_sale     = rename_and_keep(df_ar_sale, rename_map)
df_ar_summary  = rename_and_keep(df_ar_summary, rename_map)

rename_map["ACCTNAME"] = "ชื่อเจ้าหนี้"

df_ap_purchase      = rename_and_keep(df_ap_purchase , rename_map)
df_ap_summary    = rename_and_keep(df_ap_summary  , rename_map)

In [80]:
TH_MONTHS_ABBR = [
    "ม.ค.", "ก.พ.", "มี.ค.", "เม.ย.", "พ.ค.", "มิ.ย.",
    "ก.ค.", "ส.ค.", "ก.ย.", "ต.ค.", "พ.ย.", "ธ.ค."
]

def thai_date_text(d):
    dt = pd.to_datetime(d)
    return f"{dt.day} {TH_MONTHS_ABBR[dt.month-1]} {dt.year + 543}"

def thai_be_date(d):
    if pd.isna(d):
        return ""

    dt = pd.to_datetime(d)
    return f"{dt.day:02d}/{dt.month:02d}/{dt.year + 543}"

In [81]:
for df in [df_ap_purchase, df_ap_summary, df_ar_sale, df_ar_summary]:

    if "วันที่" in df.columns:
        df["วันที่"] = df["วันที่"].apply(thai_be_date)

In [82]:
df_ap_summary["จำนวนบิล"] = df_ap_summary["จำนวนบิล"].astype(int)
df_ar_summary["จำนวนบิล"] = df_ar_summary["จำนวนบิล"].astype(int)

In [83]:
import pandas as pd
from openpyxl import Workbook
from openpyxl.styles import Font, PatternFill, Border, Side, Alignment
from openpyxl.utils import get_column_letter
from openpyxl.worksheet.table import Table, TableStyleInfo


def _export_single_report(
    detail_df,
    summary_df,
    output_file,
    *,
    cutoff,
    thai_date_text,
    report_type,  # "AR" or "AP"
):
    if report_type.upper() == "AR":
        sheets = {
            "AR": detail_df.copy(),
            "AR Summary": summary_df.copy(),
        }
        title_map = {
            "AR": "รายงานลูกหนี้คงค้าง (AR Detail)",
            "AR Summary": "สรุปลูกหนี้คงค้าง (AR Summary)",
        }
    elif report_type.upper() == "AP":
        sheets = {
            "AP": detail_df.copy(),
            "AP Summary": summary_df.copy(),
        }
        title_map = {
            "AP": "รายงานเจ้าหนี้คงค้าง (AP Detail)",
            "AP Summary": "สรุปเจ้าหนี้คงค้าง (AP Summary)",
        }
    else:
        raise ValueError("report_type must be either 'AR' or 'AP'")

    wb = Workbook()
    wb.remove(wb.active)

    # styles
    header_fill = PatternFill("solid", fgColor="1F4E78")
    header_font = Font(name="Tahoma", size=12, bold=True, color="FFFFFF")
    body_font = Font(name="Tahoma", size=11, color="000000")
    total_font = Font(name="Tahoma", size=11, bold=True, color="000000")
    title_font = Font(name="Tahoma", size=14, bold=True)
    subtitle_font = Font(name="Tahoma", size=11)

    thin_gray = Side(style="thin", color="D9D9D9")
    header_border = Border(bottom=Side(style="medium", color="FFFFFF"))
    body_border = Border(bottom=thin_gray)

    fill_group_1 = PatternFill("solid", fgColor="F7FBFF")
    fill_group_2 = PatternFill("solid", fgColor="E8F1FB")
    total_fill = PatternFill("solid", fgColor="FFF2CC")

    header_align = Alignment(horizontal="center", vertical="center")
    text_align = Alignment(horizontal="left", vertical="center")
    num_align = Alignment(horizontal="right", vertical="center")
    date_align = Alignment(horizontal="center", vertical="center")

    amount_headers = {"จำนวน", "ยอดรวม", "DUEAMT", "total_amount", "VAT"}
    count_headers = {"จำนวนบิล", "bills"}
    date_headers = {"วันที่", "BILLDATE", "VOUCDATE2"}

    report_date = thai_date_text(cutoff - pd.Timedelta(days=1))

    title_row = 1
    subtitle_row = 2
    header_row = 4
    data_start_row = header_row + 1

    for sheet_name, df in sheets.items():
        df = df.copy()
        df.columns = df.columns.map(str)

        ws = wb.create_sheet(title=sheet_name)
        title = title_map.get(sheet_name, sheet_name)

        # title
        title_cell = ws.cell(row=title_row, column=1, value=title)
        title_cell.font = title_font
        title_cell.alignment = Alignment(horizontal="center", vertical="center")

        date_cell = ws.cell(row=subtitle_row, column=1, value=f"ณ วันที่ {report_date}")
        date_cell.font = subtitle_font
        date_cell.alignment = Alignment(horizontal="center", vertical="center")

        # merge title/subtitle
        if len(df.columns) > 0:
            ws.merge_cells(
                start_row=title_row,
                start_column=1,
                end_row=title_row,
                end_column=len(df.columns),
            )
            ws.merge_cells(
                start_row=subtitle_row,
                start_column=1,
                end_row=subtitle_row,
                end_column=len(df.columns),
            )

        # write header + data
        rows_to_write = [list(df.columns)] + df.values.tolist()

        for r_idx, row in enumerate(rows_to_write, start=header_row):
            for c_idx, value in enumerate(row, start=1):
                cell = ws.cell(row=r_idx, column=c_idx, value=value)

                if r_idx == header_row:
                    cell.fill = header_fill
                    cell.font = header_font
                    cell.alignment = header_align
                    cell.border = header_border
                else:
                    cell.font = body_font
                    cell.border = body_border

        # alternate row color by name group
        name_cols = ["ชื่อลูกหนี้", "ชื่อเจ้าหนี้", "ACCTNAME"]
        name_col = next((c for c in name_cols if c in df.columns), None)

        if name_col is not None and len(df) > 0:
            current_fill = fill_group_1
            prev_name = None
            name_col_idx = list(df.columns).index(name_col) + 1

            for row_idx in range(data_start_row, data_start_row + len(df)):
                current_name = ws.cell(row=row_idx, column=name_col_idx).value

                if prev_name is None:
                    prev_name = current_name
                elif current_name != prev_name:
                    current_fill = fill_group_2 if current_fill == fill_group_1 else fill_group_1
                    prev_name = current_name

                for col_idx in range(1, len(df.columns) + 1):
                    ws.cell(row=row_idx, column=col_idx).fill = current_fill

        # detect total row
        total_row_idx = None
        if len(df) > 0 and name_col is not None:
            for i, v in enumerate(df[name_col].astype(str), start=data_start_row):
                if v.strip().upper() == "TOTAL":
                    total_row_idx = i
                    break

        # format body cells by column type
        for col_idx, col_name in enumerate(df.columns, start=1):
            for row_idx in range(data_start_row, data_start_row + len(df)):
                cell = ws.cell(row=row_idx, column=col_idx)

                if col_name in date_headers:
                    cell.alignment = date_align

                elif col_name in amount_headers:
                    cell.number_format = '#,##0.00;[Red](#,##0.00);-'
                    cell.alignment = num_align

                elif col_name in count_headers:
                    cell.number_format = '0'
                    cell.alignment = num_align

                elif pd.api.types.is_numeric_dtype(df[col_name]):
                    cell.number_format = '#,##0.00;[Red](#,##0.00);-'
                    cell.alignment = num_align

                else:
                    cell.alignment = text_align

            if total_row_idx is not None:
                ws.cell(total_row_idx, col_idx).font = total_font
                ws.cell(total_row_idx, col_idx).fill = total_fill

        # row heights
        ws.row_dimensions[title_row].height = 24
        ws.row_dimensions[subtitle_row].height = 20
        ws.row_dimensions[header_row].height = 22
        for r in range(data_start_row, data_start_row + len(df)):
            ws.row_dimensions[r].height = 20

        # column widths
        for col_idx, col_name in enumerate(df.columns, start=1):
            col_letter = get_column_letter(col_idx)
            max_len = len(str(col_name))

            for row_idx in range(title_row, data_start_row + len(df)):
                val = ws.cell(row=row_idx, column=col_idx).value
                if val is None:
                    continue

                text = str(val)
                visual_len = len(text) + sum(1 for ch in text if ord(ch) > 127)
                max_len = max(max_len, visual_len)

            if col_name in ["ชื่อลูกหนี้", "ชื่อเจ้าหนี้", "ACCTNAME"]:
                ws.column_dimensions[col_letter].width = min(max_len + 8, 90)
            elif col_name in ["เลขที่บิล", "BILLNO"]:
                ws.column_dimensions[col_letter].width = min(max_len + 4, 24)
            elif col_name in ["วันที่", "BILLDATE", "VOUCDATE2"]:
                ws.column_dimensions[col_letter].width = 16
            else:
                ws.column_dimensions[col_letter].width = min(max_len + 3, 22)

        # freeze below header
        ws.freeze_panes = f"A{data_start_row}"

        # add table
        if len(df.columns) > 0 and len(df) > 0:
            end_col = get_column_letter(len(df.columns))
            end_row = header_row + len(df)
            table_ref = f"A{header_row}:{end_col}{end_row}"

            safe_name = sheet_name.replace(" ", "_").replace("-", "_")
            table = Table(displayName=f"Tbl_{safe_name}", ref=table_ref)
            table.tableStyleInfo = TableStyleInfo(
                name="TableStyleMedium2",
                showFirstColumn=False,
                showLastColumn=False,
                showRowStripes=False,
                showColumnStripes=False,
            )
            ws.add_table(table)

    wb.save(output_file)


def export_ar_report(
    df_ar_sale,
    df_ar_summary,
    output_file,
    *,
    cutoff,
    thai_date_text,
):
    _export_single_report(
        detail_df=df_ar_sale,
        summary_df=df_ar_summary,
        output_file=output_file,
        cutoff=cutoff,
        thai_date_text=thai_date_text,
        report_type="AR",
    )


def export_ap_report(
    df_ap_purchase,
    df_ap_summary,
    output_file,
    *,
    cutoff,
    thai_date_text,
):
    _export_single_report(
        detail_df=df_ap_purchase,
        summary_df=df_ap_summary,
        output_file=output_file,
        cutoff=cutoff,
        thai_date_text=thai_date_text,
        report_type="AP",
    )

In [84]:
ar_folder = os.path.join(
    BASE_FOLDER,
    "KCW-Data",
    "kcw_analytics",
    "04_outputs",
    "03_AR_AP_Report",
    f"ar_report_{YEAR}_{MONTH:02}.xlsx"
)

# ⭐ create directory if missing
os.makedirs(os.path.dirname(ar_folder), exist_ok=True)

ap_folder = os.path.join(
    BASE_FOLDER,
    "KCW-Data",
    "kcw_analytics",
    "04_outputs",
    "03_AR_AP_Report",
    f"ap_report_{YEAR}_{MONTH:02}.xlsx"
)

# ⭐ create directory if missing
os.makedirs(os.path.dirname(ar_folder), exist_ok=True)

In [85]:
output_file = folder

export_ar_report(
    df_ar_sale=df_ar_sale,
    df_ar_summary=df_ar_summary,
    output_file=ar_folder,
    cutoff=cutoff,
    thai_date_text=thai_date_text,
)

export_ap_report(
    df_ap_purchase=df_ap_purchase,
    df_ap_summary=df_ap_summary,
    output_file=ap_folder,
    cutoff=cutoff,
    thai_date_text=thai_date_text,
)